# 4. Model registry: which penguin model is live?

## Learning objectives

- Register model versions with GTO (Git Tag Ops), which records them in Git tags.
- Compare validation accuracy and promote the better version.
- Push every model version with DVC and restore it into an empty folder.

## Builds on Exercise 3

- The project keeps [Exercise 3](../03-dataops/dataops.ipynb)'s clean, validate and report stages, and adds split and train. It leaves out the plot stage.

## The problem

- The workshop's exercises use one project, [a study of Palmer penguins](../../project.md), and add a capability in each exercise.
- The analysis now predicts sex from bill length, bill depth, flipper length, body mass and species.
- `project` has a trained model, but no registered versions and no version marked as live.

## What you'll do

1. Step 1: register the first model, trained with `C: 0.01`.
2. Step 2: train with `C: 1`, commit and register the second model.
3. Step 3: compare their validation accuracy and assign the better version to prod.
4. Step 4: push every version and restore each into an empty folder.
5. Run the check.

- The exercise takes about 15 to 20 minutes.
- Only reading? Each step shows what its commands print.

## The rule

- A registered model version is a curated pet. It lives in the registry: its GTO tag in Git, its bytes in the DVC bucket.
- A copy of the model loaded on a machine to run it is cattle. Restore it from the registry whenever you need it.
- Git holds what makes a model: the data pointer, the code, `params.yaml`, `dvc.yaml`, `dvc.lock` and `artifacts.yaml`.
- `dvc repro` builds the splits, the model and the metrics. DVC stores them in the bucket, and `dvc.lock` records their hashes.
- GTO records which versions exist and which one is live. It needs no registry server.
- A registry works the same way for a large GPU-trained model. DVC stores the weights, and GTO tracks which version is live.

## Get a terminal

## In Binder

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from a real run of these commands.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-04-registry?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.

- The terminal is on the left, already in the `project` folder. A read-only copy of this page is on the right.
- Copy and paste a command: Ctrl+C, then Ctrl+V in the terminal. On a Mac, Cmd+C and Cmd+V.
- A session is temporary and has up to 2 GB of memory. [Download anything you want to keep](../../setup.md#download-your-work).


**Editing files**

- In Binder, double-click a file in the file panel on the left to open it in JupyterLab's editor. Save with Ctrl+S, or Cmd+S on a Mac.
- Or edit it in the terminal with `nano <file>`, `vi <file>` or `vim <file>`. Binder has all three.
- nano is the simplest. Ctrl+O, then Enter, saves. Ctrl+X quits.

- On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.

- You need [conda](../../setup.md). You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-04-registry --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git registry
cd registry
conda env create --quiet --file .binder/environment.yml
conda activate repro-registry
python scripts/setup_project.py
cd project
```

- If `conda env create` says the prefix already exists, an environment with that name is left from an earlier run. The rest of the block then uses that old environment and leaves a half-built `registry` folder. Remove both, then paste the block again: `conda deactivate; conda env remove -n repro-registry -y; cd ../..; rm -rf registry`
- The block leaves you in `registry/project`, so `cd ../..` goes to the folder above `registry`. If `cd project` failed too, use `cd ..` instead.
- Every command below runs in that `project` folder.
- In a new terminal, turn conda on as in [Set up](../../setup.md), then run `conda activate repro-registry` and `cd registry/project` from the folder above `registry`.
- To start over, run `cd ..` to leave `project`, then `python scripts/setup_project.py --reset`. This deletes your work.

**Editing files**

- Use any text editor.
- If you're not sure, `nano FILE` works in the terminal. Ctrl+O, then Enter, saves. Ctrl+X quits.

**With uv instead of conda**

- The terminal starts in the exercise's conda environment, and `project` is already built. To use uv instead, build `.venv` from the exercise's `uv.lock` in the folder above `project`, and activate it:

```bash
cd ~
uv sync --locked
source .venv/bin/activate
cd project
```

- You don't run `scripts/setup_project.py`. Binder ran it when it built the session, and a second run leaves `project` as it is.
- `uv sync` downloads Python 3.12.12 and the locked packages. It took 7 to 11 seconds on mybinder.org.
- A new terminal starts in conda's environment again. Run `source ~/.venv/bin/activate` in it.

**With uv instead of conda**

- [Install uv](../../setup.md#install-uv) first. Then paste this instead of the block above:

```bash
git clone --branch binder-04-registry --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git registry
cd registry
uv sync --locked
source .venv/bin/activate
uv run python scripts/setup_project.py
cd project
```

- `uv sync --locked` downloads Python 3.12.12 itself. It also builds antlr4-python3-runtime, which DVC needs, from source. That package is pure Python, so it needs no compiler.
- Activation puts Python, DVC and GTO on your path. The page's own commands then work as written.
- If you reopen a terminal, go to the exercise folder, run `source .venv/bin/activate`, then `cd project`.

- DVC may print "Waiting to acquire lock. If DVC froze, see hardlink_lock" several times. It's harmless, and the command finishes.

## The project

- `dvc repro` runs five stages: clean, validate, report, split and train.
- The data is the published 2007 to 2009 table. Cleaning leaves 333 rows with recorded sex and body mass.
- `metrics.json` keeps the sexual dimorphism result, male mean mass minus female mean mass per species.
- `split.py` reads `results/validated.csv`. `splits.json` records 216 training row positions from 2007 and 2008 and 117 validation row positions from 2009.
- `train.py` standardizes the four measurements and one-hot species columns, then fits logistic regression.
- `params.yaml` sets `train.C`, the inverse regularization strength. `model_metrics.json` records `val_accuracy`.
- Git holds the code, `params.yaml`, `dvc.yaml`, `dvc.lock`, `artifacts.yaml` and the data pointer `data/penguins.csv.dvc`.
- The `Snakefile` from the earlier exercises is still in Git. This exercise runs the steps with `dvc repro`.
- DVC stores the data, `splits.json`, `model.pkl` and both metrics files in the bucket. DVC lists them in `.gitignore`, so `git add .` leaves them out.
- `artifacts.yaml` names the model `penguin-sex-clf` and points to `model.pkl`.
- `../osn-bucket`, a folder next to `project`, is the DVC remote. Setup has pushed the first model there.

```bash
cat params.yaml
cat model_metrics.json
git ls-files
gto show
```

In [1]:
# Recorded output. You don't need to run this.
import sys
sys.path.insert(0, ".exercise")
from page import reset, root, run
run("cat params.yaml", "cat model_metrics.json", "git ls-files", "gto show")

$ cat params.yaml
report:
  statistic: mean          # or median
train:
  C: 0.01

$ cat model_metrics.json
{
  "val_accuracy": 0.846
}

$ git ls-files
.dvc/.gitignore
.dvc/config
.dvcignore
.gitignore
README.md
Snakefile
artifacts.yaml
clean.py
data/.gitignore
data/penguins.csv.dvc
dvc.lock
dvc.yaml
params.yaml
report.py
split.py
train.py
validate.py

$ gto show
╒══════════════════╤══════════╕
│ name             │ latest   │
╞══════════════════╪══════════╡
│ *penguin-sex-clf │ -        │
╘══════════════════╧══════════╛


## Step 1: register the first version

**Problem:** the first model exists, but the registry has no versions.

**Facts**

- `C: 0.01` gives validation accuracy 0.846, about 85 percent.
- `gto register` names a committed version with a Git tag.
- Setup has already committed the first model's parameters and pipeline lock. `dvc.lock` records the hash of the model and its metrics. The files are in the bucket.
- gto prints a `git push` hint after each register and assign. This project has no Git remote, so ignore the hint.

**Steps**

1. Register the current commit as v1.0.0 and inspect the registry.

```bash
gto register penguin-sex-clf --version v1.0.0
gto show
```

In [2]:
# Recorded output. You don't need to run this.
run("gto register penguin-sex-clf --version v1.0.0", "gto show")

$ gto register penguin-sex-clf --version v1.0.0
Created git tag 'penguin-sex-clf@v1.0.0' that registers version
To push the changes upstream, run:
    git push origin penguin-sex-clf@v1.0.0

$ gto show
╒═════════════════╤══════════╕
│ name            │ latest   │
╞═════════════════╪══════════╡
│ penguin-sex-clf │ v1.0.0   │
╘═════════════════╧══════════╛


## Step 2: train and register another version

**Problem:** one score doesn't tell you which value of C to use.

**Facts**

- Larger C means weaker regularization.
- The training and validation years stay the same for both versions.
- DVC reruns train when `train.C` changes. The analysis result in `metrics.json` stays the same.
- Commit before registering, so the version tag points to the parameters and lock you actually used. The new `dvc.lock` records the hashes of the new model and metrics.

**Steps**

1. Open `params.yaml`, change `train.C` from `0.01` to `1`, and save it.
2. Train and read the validation score. It should be 0.889.

```bash
dvc repro
cat model_metrics.json
```

In [3]:
# Recorded output. You don't need to run this.
params = root / "project" / "params.yaml"
params.write_text(params.read_text().replace("C: 0.01", "C: 1"))
run("dvc repro", "cat model_metrics.json")

$ dvc repro
'data/penguins.csv.dvc' didn't change, skipping
Stage 'clean' didn't change, skipping
Stage 'validate' didn't change, skipping
Stage 'report' didn't change, skipping
Stage 'split' didn't change, skipping
Running stage 'train':
> python train.py
{'val_accuracy': 0.889}
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.

$ cat model_metrics.json
{
  "val_accuracy": 0.889
}


3. Commit the parameters and the lock, and register v1.1.0.

```bash
git add params.yaml dvc.lock
git commit --quiet -m "Train the penguin sex classifier with C=1"
gto register penguin-sex-clf --version v1.1.0
```

In [4]:
# Recorded output. You don't need to run this.
run("git add params.yaml dvc.lock", "git commit --quiet -m \"Train the penguin sex classifier with C=1\"", "gto register penguin-sex-clf --version v1.1.0")

$ git add params.yaml dvc.lock

$ git commit --quiet -m "Train the penguin sex classifier with C=1"

$ gto register penguin-sex-clf --version v1.1.0
Created git tag 'penguin-sex-clf@v1.1.0' that registers version
To push the changes upstream, run:
    git push origin penguin-sex-clf@v1.1.0


## Step 3: promote the better validation score

**Problem:** both versions are registered, but neither is marked as live.

**Facts**

- Compare `val_accuracy` in `model_metrics.json`.
- `dvc metrics show --all-tags` and `dvc metrics diff <tag>` read each tag's metrics by the hash in its `dvc.lock`. DVC takes them from the cache, or from the bucket if the cache doesn't have them. With a real bucket, they work in a fresh clone without `dvc pull`.
- v1.1.0 scores 0.889, above v1.0.0 at 0.846.
- `gto assign` records a stage in a Git tag. Here, `prod` means the version chosen to be live.
- Assigning prod records a choice. It doesn't deploy a service.

**Steps**

1. Compare the registered versions, then the current result with the first version.

```bash
dvc metrics show --all-tags model_metrics.json
dvc metrics diff penguin-sex-clf@v1.0.0 --targets model_metrics.json
```

In [5]:
# Recorded output. You don't need to run this.
run("dvc metrics show --all-tags model_metrics.json", "dvc metrics diff penguin-sex-clf@v1.0.0 --targets model_metrics.json")

$ dvc metrics show --all-tags model_metrics.json
Revision                Path                val_accuracy
workspace               model_metrics.json  0.889
penguin-sex-clf@v1.0.0  model_metrics.json  0.846
penguin-sex-clf@v1.1.0  model_metrics.json  0.889

$ dvc metrics diff penguin-sex-clf@v1.0.0 --targets model_metrics.json
Path                Metric        penguin-sex-clf@v1.0.0    workspace    Change
model_metrics.json  val_accuracy  0.846                     0.889        0.043


2. Promote v1.1.0 and inspect the record.

```bash
gto assign penguin-sex-clf --version v1.1.0 --stage prod
gto show
gto history penguin-sex-clf
```

In [6]:
# Recorded output. You don't need to run this.
run("gto assign penguin-sex-clf --version v1.1.0 --stage prod", "gto show", "gto history penguin-sex-clf")

$ gto assign penguin-sex-clf --version v1.1.0 --stage prod
Created git tag 'penguin-sex-clf#prod#1' that assigns stage to version 'v1.1.0'
To push the changes upstream, run:
    git push origin 'penguin-sex-clf#prod#1'

$ gto show
╒═════════════════╤══════════╤═════════╕
│ name            │ latest   │ #prod   │
╞═════════════════╪══════════╪═════════╡
│ penguin-sex-clf │ v1.1.0   │ v1.1.0  │
╘═════════════════╧══════════╧═════════╛

$ gto history penguin-sex-clf
╒═══════════════════════════╤═════════════════╤══════════════╤═══════════╤═════════╤══════════╤════════════════════════╕
│ timestamp                 │ artifact        │ event        │ version   │ stage   │ commit   │ ref                    │
╞═══════════════════════════╪═════════════════╪══════════════╪═══════════╪═════════╪══════════╪════════════════════════╡
│ 2026-11-08 21:17:16+00:00 │ penguin-sex-clf │ assignment   │ v1.1.0    │ prod    │ 07785ed  │ penguin-sex-clf#prod#1 │
│ 2026-11-08 21:17:15+00:00 │ penguin-sex-clf │ r

- Your timestamps and your v1.1.0 commit ID will differ from these. They record when you ran the commands.
- `gto history` sorts by time, to the second. Rows from the same second can print in a different order than here.

## Step 4: push and restore every version

**Problem:** a version tag doesn't contain the model's bytes.

**Facts**

- `dvc push` sends the current version's model, splits and metrics to the bucket.
- The bucket here is a folder, `../osn-bucket`, on the same machine. It stands in for an S3 bucket such as one on the Open Storage Network (OSN). With a real bucket, you install DVC's S3 support, `dvc-s3`, and set the remote's URL, endpoint and keys. The DVC commands stay the same.
- `dvc push --all-tags` also sends the files referenced by older tags.
- Setup already pushed v1.0.0. Push all tags whenever you release, so older versions remain available.
- `restore.py` clones each registered tag into a separate empty folder and pulls from the bucket. Those folders don't use your local DVC cache.
- With a Git hosting service, you'd also push your commits and tags. This local exercise uses no Git remote.

**Steps**

1. Push the current version, then every tagged version.

```bash
dvc push
dvc push --all-tags
```

In [7]:
# Recorded output. You don't need to run this.
run("dvc push", "dvc push --all-tags")

$ dvc push
2 files pushed

$ dvc push --all-tags
Everything is up to date.


2. Restore both registered models into empty folders.

```bash
python ../scripts/restore.py
```

In [8]:
# Recorded output. You don't need to run this.
run("python ../scripts/restore.py")

$ python ../scripts/restore.py
penguin-sex-clf@v1.0.0: restored into an empty folder
penguin-sex-clf@v1.1.0: restored into an empty folder


- Both `penguin-sex-clf@v1.0.0` and `penguin-sex-clf@v1.1.0` should say `restored into an empty folder`.

## Check your work

- Run this when all four steps are done. Each line says what to do next.
- The check reads each registered tag's parameters from Git and its validation accuracy through DVC.
- It requires two versions with different C values and prod at the highest validation accuracy.
- It restores the `prod` version into a clean clone, rebuilds it with `dvc repro --force`, and compares its validation accuracy with the recorded one.
- It restores every registered version into a separate empty folder.

```bash
python ../check.py
```

On the starter project it fails like this:

In [10]:
# Recorded output. You don't need to run this.
reset()
run("python ../check.py")

$ python ../check.py
FAIL  Register v1.0.0, train and commit a different C, then register v1.1.0 (steps 1 and 2).
FAIL  Run dvc metrics show --all-tags, then use gto assign penguin-sex-clf --version <best> --stage prod (step 3).
FAIL  Assign a registered version to prod before checking its clean rebuild.
FAIL  Register the versions and run dvc push --all-tags. Missing: no registered versions

0 of 4 parts pass.


## Summary

- GTO names committed model versions and records which version is live.
- Compare validation scores before assigning prod.
- Git holds the code, parameters and `dvc.lock`. DVC stores the splits, model and metrics, and reads any version's metrics with `dvc metrics show --all-tags`.
- A registered version is a curated pet in the registry. A copy loaded on a machine is cattle, restored from the bucket whenever you need it.
- Push every tagged version, and check that each one restores into an empty folder.

## On your own project

- Register the last model you shared, with its parameters and `dvc.lock` committed and its files pushed.
- Choose a version using your project's validation measure.
- Restore that version on a clean machine before you depend on it.

## About the data

- Palmer penguins, CC0, 344 rows collected in 2007 to 2009. The exercise uses the published table unchanged.
- Data: palmerpenguins (Horst, Hill and Gorman 2020, doi:10.5281/zenodo.3960218), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.

## When you're done

From the folder above `registry`:

```bash
conda deactivate
conda env remove --yes --name repro-registry
rm -rf registry
```

- With uv, run `deactivate` instead of the two conda lines. `rm -rf registry` removes `.venv` too.

The [Literate Science Results](../../results/results.ipynb) page runs this project's scripts on all three seasons, at both values of C, each time the site is published.